In [ ]:
# =====================================================================
# BOLUM 0 - Ayarlar, sabitler ve veri yukleme
# ONEMLI: Bu asamada hicbir model egitilmez, hicbir metrik hesaplanmaz.
# =====================================================================
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

RANDOM_STATE = 42          # tum rastgelelik iceren islemler icin sabit
TEST_SIZE    = 0.20        # test kumesi orani (geometri bazinda)
N_SPLITS     = 5           # GroupKFold kat sayisi
ROUND_DEC    = 6           # Geometry_ID uretiminde float yuvarlama basamagi
CV_UZERINDE  = "train"     # "train" -> CV yalnizca egitim kumesinde | "all" -> tum veri

# ---- Portable project paths ---------------------------------------------
from pathlib import Path

def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "masonry_tower_primary_dataset.xlsx").is_file():
            return candidate
    raise FileNotFoundError(
        "Project root could not be located. Run this notebook from the repository "
        "root or from its code/ directory, and keep the data/ directory unchanged."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAT_FILE = DATA_DIR / "masonry_tower_primary_dataset.xlsx"
# ------------------------------------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Degisken tanimlari (sira sabittir, ileriki asamalarda aynen korunacaktir)
GEO_COLS = ["Height (m)", "Section a (m)", "Section b (m)", "Wall Thickness (m)",
            "Opening z/H", "Opening Ratio x (%)", "Opening Ratio y (%)"]
MAT_COLS = ["E (MPa)", "d (kg/m3)"]
FEATURE_COLS = GEO_COLS + MAT_COLS          # X matrisi (9 ozellik)
TARGET_1 = "f1 (Hz)"
TARGET_2 = "f2 (Hz)"

df = pd.read_excel(MAT_FILE, sheet_name=0)
print(f"Veri yuklendi: {df.shape[0]} satir x {df.shape[1]} sutun")

In [ ]:
# =====================================================================
# BOLUM 1 - Geometry_ID olusturulmasi ve grup yapisinin dogrulanmasi
# =====================================================================

def geometry_id_olustur(veri, geo_cols=GEO_COLS, ndec=ROUND_DEC):
    """Yedi geometrik degiskenden benzersiz Geometry_ID uretir (G001, G002, ...)."""
    anahtar = veri[geo_cols].round(ndec).astype(str).agg("|".join, axis=1)
    esleme = {k: f"G{i+1:03d}" for i, k in enumerate(pd.unique(anahtar))}
    return anahtar.map(esleme)


def geometry_id_raporu(veri, beklenen_geometri=191, beklenen_kayit=6):
    """Her Geometry_ID icin kayit ve malzeme kombinasyonu sayisini raporlar."""
    ozet = (veri.groupby("Geometry_ID")
                .apply(lambda g: pd.Series({
                    "Record_count": len(g),
                    "Unique_material_cases": g[MAT_COLS].drop_duplicates().shape[0],
                }), include_groups=False)
                .reset_index())

    dagilim = (ozet["Record_count"].value_counts().sort_index()
               .rename_axis("Record_count").reset_index(name="Number_of_geometries"))

    duzensiz = ozet[(ozet["Record_count"] != beklenen_kayit) |
                    (ozet["Unique_material_cases"] != beklenen_kayit)]

    kontrol = pd.DataFrame([
        {"Check": "Total number of rows",
         "Expected": beklenen_geometri * beklenen_kayit, "Observed": len(veri)},
        {"Check": "Number of unique Geometry_IDs",
         "Expected": beklenen_geometri, "Observed": veri["Geometry_ID"].nunique()},
        {"Check": "Geometries with exactly 6 records",
         "Expected": beklenen_geometri, "Observed": int((ozet["Record_count"] == 6).sum())},
        {"Check": "Geometries with 6 unique material cases",
         "Expected": beklenen_geometri, "Observed": int((ozet["Unique_material_cases"] == 6).sum())},
    ])
    kontrol["Status"] = np.where(kontrol["Expected"] == kontrol["Observed"], "OK", "CHECK")
    return ozet, dagilim, duzensiz, kontrol


df["Geometry_ID"] = geometry_id_olustur(df)
ozet_geo, dagilim_geo, duzensiz_geo, kontrol_geo = geometry_id_raporu(df)

print("\n--- Geometry_ID kontrolu ---")
print(kontrol_geo.to_string(index=False))
if (kontrol_geo["Status"] == "CHECK").any():
    print("UYARI: Beklenen grup yapisi saglanmiyor. Ayrintilar rapor dosyasinda.")

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Geometry_ID_Report.xlsx"),
                    engine="openpyxl") as writer:
    kontrol_geo.to_excel(writer, sheet_name="Validation_Checks", index=False)
    dagilim_geo.to_excel(writer, sheet_name="Record_count_distribution", index=False)
    ozet_geo.to_excel(writer, sheet_name="Geometry_summary", index=False)
    duzensiz_geo.to_excel(writer, sheet_name="Irregular_geometries", index=False)

In [ ]:
# =====================================================================
# BOLUM 2 - X, y1, y2 tanimi ve guvenlik kontrolleri
# =====================================================================

X  = df[FEATURE_COLS].copy()      # 9 ozellik, sira sabit
y1 = df[TARGET_1].copy()
y2 = df[TARGET_2].copy()
groups = df["Geometry_ID"].copy() # yalnizca gruplama icin, ozellik degildir

# Guvenlik kontrolleri: Geometry_ID ve hedefler X icinde bulunmamalidir
yasakli = ["Geometry_ID", TARGET_1, TARGET_2]
sizinti = [c for c in yasakli if c in X.columns]
if sizinti:
    raise ValueError(f"HATA: X matrisinde bulunmamasi gereken sutunlar var: {sizinti}")
if X.isna().sum().sum() or y1.isna().sum() or y2.isna().sum():
    raise ValueError("HATA: X veya hedeflerde eksik deger bulundu.")

print(f"\nX matrisi : {X.shape[0]} satir x {X.shape[1]} ozellik")
print("Ozellik sirasi:", list(X.columns))
print(f"Hedefler  : {TARGET_1}, {TARGET_2} | Grup degiskeni: Geometry_ID (ozellik degil)")

In [ ]:
# =====================================================================
# BOLUM 3 - Geometri bazli egitim-test ayrimi (%80 / %20)
# =====================================================================

def grup_bazli_bolme(gruplar, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """Ayni geometrinin tum kayitlarini ayni tarafta tutan tek bir train/test ayrimi uretir."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(np.zeros(len(gruplar)), groups=gruplar))
    return train_idx, test_idx


def sizinti_kontrolu(gruplar, train_idx, test_idx):
    """Egitim ve test kumelerinde ortak geometri bulunursa hata uretir."""
    ortak = set(gruplar.iloc[train_idx]) & set(gruplar.iloc[test_idx])
    if ortak:
        raise ValueError(f"VERI SIZINTISI: {len(ortak)} geometri hem egitimde hem testte "
                         f"bulunuyor. Ornek: {sorted(ortak)[:5]}")
    if len(set(train_idx) & set(test_idx)) > 0:
        raise ValueError("VERI SIZINTISI: Ortak satir indeksi bulundu.")
    print("Sizinti kontrolu: OK - ortak geometri yok.")


train_idx, test_idx = grup_bazli_bolme(groups)
sizinti_kontrolu(groups, train_idx, test_idx)

df["Split"] = "Train"
df.iloc[test_idx, df.columns.get_loc("Split")] = "Test"

# Ozet tablo
bolme_ozet = pd.DataFrame([
    {"Set": "Train", "Number_of_records": len(train_idx),
     "Record_percentage_%": round(100 * len(train_idx) / len(df), 2),
     "Number_of_geometries": groups.iloc[train_idx].nunique(),
     "Geometry_percentage_%": round(100 * groups.iloc[train_idx].nunique() / groups.nunique(), 2)},
    {"Set": "Test", "Number_of_records": len(test_idx),
     "Record_percentage_%": round(100 * len(test_idx) / len(df), 2),
     "Number_of_geometries": groups.iloc[test_idx].nunique(),
     "Geometry_percentage_%": round(100 * groups.iloc[test_idx].nunique() / groups.nunique(), 2)},
])

# Bolmenin temsil gucunu gormek icin hedef araliklari (metrik degildir, tanimlayici bilgidir)
hedef_kontrol = (df.groupby("Split")[[TARGET_1, TARGET_2]]
                   .agg(["count", "mean", "std", "min", "max"]).round(4))

sizinti_dogrulama = pd.DataFrame([{
    "Check": "Common geometries between Train and Test",
    "Expected": 0,
    "Observed": len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])),
    "Status": "OK",
}])

print("\n--- Egitim-Test ayrimi ---")
print(bolme_ozet.to_string(index=False))

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Train_Test_Summary.xlsx"),
                    engine="openpyxl") as writer:
    bolme_ozet.to_excel(writer, sheet_name="Split_summary", index=False)
    sizinti_dogrulama.to_excel(writer, sheet_name="Leakage_check", index=False)
    hedef_kontrol.to_excel(writer, sheet_name="Target_range_by_split")
    (df.loc[df["Split"] == "Train", "Geometry_ID"].drop_duplicates()
       .reset_index(drop=True).to_frame("Train_Geometry_ID")
       .to_excel(writer, sheet_name="Train_geometries", index=False))
    (df.loc[df["Split"] == "Test", "Geometry_ID"].drop_duplicates()
       .reset_index(drop=True).to_frame("Test_Geometry_ID")
       .to_excel(writer, sheet_name="Test_geometries", index=False))

In [ ]:
# =====================================================================
# BOLUM 4 - 5 katli GroupKFold yapisi
# Not: GroupKFold deterministiktir (rastgelelik icermez). Yine de olusan kat
# atamasi diske yazilarak tum modellerde ayni yapinin kullanimi garanti edilir.
# =====================================================================

# CV'nin uygulanacagi alt kume
if CV_UZERINDE == "train":
    cv_mask = (df["Split"] == "Train").values
else:
    cv_mask = np.ones(len(df), dtype=bool)

X_cv      = X.loc[cv_mask].reset_index(drop=True)
groups_cv = groups.loc[cv_mask].reset_index(drop=True)
cv_orijinal_indeks = df.index[cv_mask]          # atamayi ana tabloya geri yazmak icin


def groupkfold_yapisi(X_alt, gruplar_alt, n_splits=N_SPLITS):
    """5 katli GroupKFold uretir; her fold icin ozet ve kat atamasi dondurur."""
    gkf = GroupKFold(n_splits=n_splits)
    satirlar, atama = [], pd.Series(np.nan, index=X_alt.index, dtype="float")

    for fold, (tr, va) in enumerate(gkf.split(X_alt, groups=gruplar_alt), start=1):
        atama.iloc[va] = fold
        satirlar.append({
            "Fold": fold,
            "Train_samples": len(tr),
            "Validation_samples": len(va),
            "Train_geometries": gruplar_alt.iloc[tr].nunique(),
            "Validation_geometries": gruplar_alt.iloc[va].nunique(),
            "Overlapping_geometries": len(set(gruplar_alt.iloc[tr]) & set(gruplar_alt.iloc[va])),
        })
    return pd.DataFrame(satirlar), atama.astype(int)


def fold_bagimsizlik_kontrolu(gruplar_alt, atama, n_splits=N_SPLITS):
    """Her geometrinin tek bir validation fold'unda yer aldigini dogrular."""
    # 1) Fold ici sizinti
    fold_gruplari = {f: set(gruplar_alt[atama == f]) for f in range(1, n_splits + 1)}

    # 2) Fold'lar arasi kesisim bos olmali
    kesisimler = []
    for i in range(1, n_splits + 1):
        for j in range(i + 1, n_splits + 1):
            ortak = fold_gruplari[i] & fold_gruplari[j]
            kesisimler.append({"Fold_A": i, "Fold_B": j, "Common_geometries": len(ortak)})
            if ortak:
                raise ValueError(f"HATA: Fold {i} ve Fold {j} ortak geometri iceriyor.")

    # 3) Her geometri tam olarak bir kez validation olmali
    kez = gruplar_alt.groupby(gruplar_alt).apply(lambda s: atama.loc[s.index].nunique())
    if (kez != 1).any():
        raise ValueError("HATA: Bazi geometriler birden fazla validation fold'una dagilmis.")

    # 4) Tum geometriler kapsanmali
    kapsanan = set().union(*fold_gruplari.values())
    if kapsanan != set(gruplar_alt.unique()):
        raise ValueError("HATA: Bazi geometriler hicbir validation fold'unda yer almiyor.")

    print("Fold bagimsizlik kontrolu: OK - fold'lar ayrik ve tum geometriler kapsanmis.")
    return pd.DataFrame(kesisimler)


cv_ozet, fold_atama = groupkfold_yapisi(X_cv, groups_cv)
kesisim_tablosu = fold_bagimsizlik_kontrolu(groups_cv, fold_atama)

# Fold atamasini ana tabloya yaz (test satirlari icin bos kalir)
df["CV_Fold"] = np.nan
df.loc[cv_orijinal_indeks, "CV_Fold"] = fold_atama.values

# Fold basina geometri listesi
fold_geometrileri = (pd.DataFrame({"Geometry_ID": groups_cv, "Validation_Fold": fold_atama})
                     .drop_duplicates().sort_values(["Validation_Fold", "Geometry_ID"]))

print("\n--- GroupKFold ozeti ---")
print(cv_ozet.to_string(index=False))

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "CrossValidation_Summary.xlsx"),
                    engine="openpyxl") as writer:
    cv_ozet.to_excel(writer, sheet_name="Fold_summary", index=False)
    kesisim_tablosu.to_excel(writer, sheet_name="Fold_independence_check", index=False)
    fold_geometrileri.to_excel(writer, sheet_name="Geometry_fold_assignment", index=False)
    pd.DataFrame([{"Setting": "CV applied on", "Value": CV_UZERINDE},
                  {"Setting": "n_splits", "Value": N_SPLITS},
                  {"Setting": "Group variable", "Value": "Geometry_ID"},
                  {"Setting": "random_state", "Value": RANDOM_STATE}]
                 ).to_excel(writer, sheet_name="CV_settings", index=False)

In [ ]:
# =====================================================================
# BOLUM 5 - Olcekleme stratejisi (bu asamada UYGULANMAZ, yalnizca tablo)
# =====================================================================

OLCEKLEME_STRATEJISI = [
    {"Model": "Linear Regression", "Scaling_Required": "No",
     "Reason": "Ordinary least squares is invariant to feature scaling",
     "Pipeline_Structure": "Model"},
    {"Model": "Ridge Regression", "Scaling_Required": "Yes",
     "Reason": "L2 penalty depends on the magnitude of the coefficients",
     "Pipeline_Structure": "StandardScaler -> Model"},
    {"Model": "Support Vector Regression", "Scaling_Required": "Yes",
     "Reason": "Kernel functions are distance based",
     "Pipeline_Structure": "StandardScaler -> Model"},
    {"Model": "K-Nearest Neighbors Regression", "Scaling_Required": "Yes",
     "Reason": "Neighbour search is distance based",
     "Pipeline_Structure": "StandardScaler -> Model"},
    {"Model": "Decision Tree Regression", "Scaling_Required": "No",
     "Reason": "Split thresholds are independent of feature scale",
     "Pipeline_Structure": "Model"},
    {"Model": "Random Forest Regression", "Scaling_Required": "No",
     "Reason": "Split thresholds are independent of feature scale",
     "Pipeline_Structure": "Model"},
    {"Model": "Gradient Boosting Regression", "Scaling_Required": "No",
     "Reason": "Split thresholds are independent of feature scale",
     "Pipeline_Structure": "Model"},
]

olcekleme_df = pd.DataFrame(OLCEKLEME_STRATEJISI)
print("\n--- Olcekleme stratejisi ---")
print(olcekleme_df[["Model", "Scaling_Required"]].to_string(index=False))

not_df = pd.DataFrame([
    {"Note": "StandardScaler must be fitted inside the cross-validation loop, "
             "never on the full dataset, to avoid information leakage."},
    {"Note": "Scaling is not applied in this stage; only the strategy is defined."},
])

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Scaling_Strategy.xlsx"),
                    engine="openpyxl") as writer:
    olcekleme_df.to_excel(writer, sheet_name="Scaling_Strategy", index=False)
    not_df.to_excel(writer, sheet_name="Notes", index=False)

In [ ]:
# =====================================================================
# BOLUM 6 - Pipeline yapisinin tanimlanmasi (CALISTIRILMAZ) ve bolmenin kaydi
# UYARI: Asagida hicbir fit / predict cagrisi yoktur.
# =====================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def pipeline_olustur(model, olcekleme_gerekli):
    """Gerekiyorsa StandardScaler ekleyerek Pipeline nesnesi kurar (egitmez)."""
    adimlar = []
    if olcekleme_gerekli:
        adimlar.append(("scaler", StandardScaler()))
    adimlar.append(("model", model))
    return Pipeline(adimlar)


def pipeline_yapisini_goster(model_adi, pipe):
    """Pipeline adimlarini okunabilir bicimde yazdirir."""
    print(f"{model_adi:32s} -> " + " | ".join(ad for ad, _ in pipe.steps))


# Ornek kullanim: modeller ileriki asamada tanimlanacaktir.
# from sklearn.linear_model import Ridge
# pipe_ridge = pipeline_olustur(Ridge(random_state=RANDOM_STATE), olcekleme_gerekli=True)
# pipeline_yapisini_goster("Ridge Regression", pipe_ridge)
print("\nPipeline fabrikasi tanimlandi (hicbir model kurulmadi/egitilmedi).")

# ---- Bolmenin dondurulmasi -------------------------------------------
bolme_atamasi = df[["Geometry_ID", "Split", "CV_Fold"]].copy()
bolme_atamasi.insert(0, "Row_index", df.index)

with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "Data_Split_Assignment.xlsx"),
                    engine="openpyxl") as writer:
    bolme_atamasi.to_excel(writer, sheet_name="Row_assignment", index=False)
    pd.DataFrame({"Feature_order": FEATURE_COLS}).to_excel(
        writer, sheet_name="Feature_order", index=False)
    pd.DataFrame([{"Setting": "random_state", "Value": RANDOM_STATE},
                  {"Setting": "test_size", "Value": TEST_SIZE},
                  {"Setting": "n_splits", "Value": N_SPLITS},
                  {"Setting": "split_method", "Value": "GroupShuffleSplit"},
                  {"Setting": "cv_method", "Value": "GroupKFold"},
                  {"Setting": "group_variable", "Value": "Geometry_ID"}]
                 ).to_excel(writer, sheet_name="Split_settings", index=False)

print("\nAsama 3 tamamlandi. Hicbir model egitilmedi, hicbir metrik hesaplanmadi.")
print("Cikti klasoru:", OUTPUT_DIR)